# __STIR FUTURE LANDING ZONES FAIR VALUE GRID__

### Notebook overview:

The aim of this notebook is to estimate front-end futures landing zones given a range of path assumptios provided by the user. The note book is divided into different parts:

- General packages installation and auxiliary functions used to run the code
- Downloading CB meetings and holiday calendar
- Downloading futures contract specifications, including effective dates, %of the meetings falling within the life of the contract and basis adjustment.
- Downloading OIS O/Rate for compounding purposes
- Estimaation of landing zones for the contract
- Data export into excel

The procedure uses OIS O/N rate compounding for a given rate path assumption. The code iterates across all the ranges provided
returning a grid of fair values.

## Installing Packages

#### Jupyter Notebook Import:

In [ ]:
import blpapi
import matplotlib.pyplot as plt
import pdblp
import datetime as dt
from datetime import timedelta as timedelta
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools as it
import calendar
from pandas.tseries.holiday import get_calendar, HolidayCalendarFactory
from pandas.tseries.offsets import CustomBusinessDay
import itertools
from joblib import Parallel, delayed
from collections import defaultdict
from itertools import product
import plotly.graph_objects as go

### BBG- Data Retrival Functions:

In [ ]:
def load_srch_results(srch_name):
    """This functions mimics BRSCH in excel. Takes as only input the
    name of the SRCH as a string, returns the result of the query as a list """
    const = "FI:"
    search = const + srch_name
    res = con.bsrch(search)
    return res[res.columns[0]].values.tolist()

def bdp(bonds_list, fld):
    """This functions mimics BDP in excel. Takes as input a list of ticker and the
    the name of the field as a string"""
    tmp = con.ref(bonds_list, [fld])
    tmp = tmp.set_index('ticker').drop(columns = ['field'])
    tmp.columns = [fld]
    return tmp

def bbg_download_clean_df(ticker_list, fields, start_date, end_date):
    start_date, end_date = start_date.strftime('%Y%m%d'), end_date.strftime('%Y%m%d')
    tmp = con.bdh(ticker_list, fields, start_date, end_date)
    tmp = tmp[ticker_list]
    tmp.columns = tmp.columns.droplevel(1)
    tmp = tmp.sort_values(by ='date', ascending = False)
    return tmp

### Bloomberg API Connect:

In [ ]:
con = pdblp.BCon(debug=False, port=8194, timeout = 50000)

In [ ]:
con.start()

## Auxiliary Functions:

In [ ]:
#########################################################################
def adjust_effective_Date(df, target2_holidays):
    """
    Adjust the 'Effective_date column of a CB meeting df. 
    If a date is  a TARGET2 hol. or a weekend returns the previous 
    available business day.
    """
    target2_bday =  CustomBusinessDay(holidays = target2_holidays)
    
    def adjust(date):
        original = date
        while (date in target2_holidays) or (date.weekday() >= 5):
            date -= pd.Timedelta(days = 1)
        if date != original:
            print(f"Adjusted {original.date()} ->{date.date()} ")
        return date
    df['Effective_date'] = df['Effective_date'].apply(adjust)
    return df

########################################################################
def er_month_mapping(month):
    if month == 3:
        er = 'H'
    elif month == 6:
        er = 'M'
    elif month == 9:
        er = 'U'
    else:
        er = 'Z'
    return er

#########################################################################
def filter_number_greater_than_current(number, list_):
    return [i for i in list_ if i >= number]

#########################################################################
def er_contracts_generation(date, prefix = 'ER' ,suffix = " Comdty"):
    """
    This function generates ER tickers on a look fwd basis
    
    """
    current_month = date.month
    current_year = date.year - 2020
    
    IMM_months = [3, 6, 9, 12]
    
    filtered_contracts_current = [(m, current_year) for m in IMM_months if m > current_month]
    filtered_contracts_next = [(m, current_year + 1) for m in IMM_months]
    
    all_contracts = filtered_contracts_current + filtered_contracts_next
    
    er_names = [prefix + er_month_mapping(m) + str(y) + suffix for m, y in all_contracts]
    
    return er_names

###############################################################################
def compute_contracts_weights(stirt_futures, nested_dates):
    
    future_start_dt = pd.to_datetime(stirt_futures['future_start_dt'])
    future_lenght_dd = stirt_futures['future_period_dd']
    
    contract_weights = [
        [((dt-future_start_dt.iloc[i]).days / future_lenght_dd.iloc[i]).__round__(2)
         for dt in sublist]
        for i, sublist in enumerate(nested_dates)
    ]
    
    return contract_weights

##################################################################################
def meetings_ranges(CB_meeting_dt, stirt_futures, verbose = True):
    
    before_future_exp = []
    after_future_exp = []
    
    today = dt.datetime.today()
    
    
    CB_meeting_dt['Effective_date'] = pd.to_datetime(CB_meeting_dt['Effective_date'])
    stirt_futures['future_start_dt'] =  pd.to_datetime(stirt_futures['future_start_dt'])
    stirt_futures['future_end_dt'] =  pd.to_datetime(stirt_futures['future_end_dt'])
    
    for idx, row in stirt_futures.iterrows():
        contract = row['ticker'] if 'ticker' in row else f" Future{idx}"
        start = row['future_start_dt']
        end = row['future_end_dt']
        
        
        before = CB_meeting_dt.loc[
            (CB_meeting_dt['Effective_date'] <= start) &
            (CB_meeting_dt['Effective_date'] > today), 
            'Effective_date'
        ].to_list()
        
        
        inside = CB_meeting_dt.loc[
            (CB_meeting_dt['Effective_date'] > start) &
            (CB_meeting_dt['Effective_date'] < end), 
            'Effective_date'
        ].to_list()
            
        before_future_exp.append(before)
        after_future_exp.append(inside)
        
        if verbose:
            print(f"\n Contract: {contract}")
            print(f"Start :{start.date()}, End: {end.date()}, Today:{today.date()}")
            print(f" Meetings Before exp: {len(before)} -> {before if before else 'None'}")
            print(f" Meetings inside exp: {len(inside)} -> {inside if inside else 'None'}")
                  
        
        
    contract_weights = compute_contracts_weights(stirt_futures, after_future_exp)
    
    return before_future_exp, after_future_exp, contract_weights

################################################################################################
def gen_df_for_estr_compunding(holidays_calendar, start_date, end_date, estr_df, effective_dt_list):
    
    target2_bday = CustomBusinessDay(holidays = holidays_calendar)
    start = start_date
    end = end_date
    
   
    
    target2_days = pd.date_range(start, end, freq = target2_bday)
    df = pd.DataFrame({'date': target2_days, 'aux':[0*i for i in range(len(target2_days))]})
    df.index = df['date']
    
    df = pd.concat([df, estr_df], axis = 1)
    df = df.drop(['date', 'aux'], axis = 'columns').reset_index()

    
    #compute how many calendar dd each rate applies to (diff target2 dd and the next)
    df['n_i'] = (df['date'].shift(-1, fill_value= end + pd.Timedelta(days= 1)) -df['date']).dt.days
    df = df.reset_index(drop = True)
    
    df['Is_Effective_date'] = df['date'].isin(effective_dt_list)
    
    return df.set_index('date')

###################################################################################################
def gen_df_with_rate_path(df, policy_path_assumption_list):
    
    effective_dates_idx = df.index[df['Is_Effective_date'] == True]
    
    if len(effective_dates_idx) != len(policy_path_assumption_list):
        raise ValueError("Number of rate cuts does not match the number of effective dates")
    
    df['path_assumption'] = float('nan')
    df.loc[df['Is_Effective_date'], 'path_assumption'] = policy_path_assumption_list
    
    for i in range(1, len(df)):
        if pd.isna(df.iloc[i]['ESTR_ON_Rate']):
            prev = df.iloc[i - 1]['ESTR_ON_Rate']
            path = df.iloc[i]['path_assumption']
        
            if pd.isna(path):
                df.iloc[i, df.columns.get_loc('ESTR_ON_Rate')] = prev
            else:
                df.iloc[i, df.columns.get_loc('ESTR_ON_Rate')] = prev + path
            
    return df
#################################################################################################
def compounded_estr(df, N = 360):
    
    d_c = df['n_i'].sum() ### TOT Calendar
    
    df['ESTR_ON_Rate'] = df['ESTR_ON_Rate']/100 ## convert from %
    
    daily_estr = df['ESTR_ON_Rate'] / N ## from ann. to daily
    
    tmp = np.prod(1+ (daily_estr * df['n_i'])) -1 
    
    compounded_rate = tmp*(N/d_c) 
    
    return compounded_rate*100 #daily_estr in %

#################################################################################################
def pull_inputs(euribor_df, contract_name):
    
    contract = euribor_df[contract_name]
    future_start_dt = contract['future_start_dt']
    future_end_dt = contract['future_end_dt']
    
    future_start_dt = future_start_dt.date()
    future_end_dt = future_end_dt.date()
    
    meetings_before_future_start = contract.loc['Eff_date_to_exp']
    meeting_inside_future_life = contract.loc['Eff_date_after_exp']
    
    eff_dates_list = contract.loc['Eff_date_to_exp'] + contract.loc['Eff_date_after_exp']
    
    
    basis = contract['Euribor-ESTR basis']
    
    return future_start_dt, future_end_dt, eff_dates_list, meetings_before_future_start ,meeting_inside_future_life, basis
####################################################################################################
def euribor_landing_zone(
    holidays_calendar, start_date, end_date, estr_df, 
    effective_dt_list, rate_path_assumption, future_start_dt
):
    
    tmp_df = gen_df_for_estr_compunding(
        holidays_calendar, start_date, end_date, estr_df, effective_dt_list
    )
    
    tmp_df = gen_df_with_rate_path(tmp_df, rate_path_assumption)
    
    #setting compunding period from the delivery of the future
    mask = tmp_df.index >= pd.Timestamp(future_start_dt)
    
    
    res = compounded_estr(tmp_df[mask][['ESTR_ON_Rate', 'n_i']])
    
    res = pd.DataFrame(res , index = [rate_path_assumption[-1]], columns = [rate_path_assumption[0]] )
    return res

################################################################################################
def unique_paths_by_sum(cut_values, num_meetings,
                          *,                 # da qui solo keyword opzionali
                          phase=None,        # "to" oppure "after" (facoltativo)
                          units="percent",   # "percent" (-0.25=-25bps) o "decimal" (-0.0025)
                          switch_bps=None,   # se impostato, per chiavi <= soglia usa right-bias
                          pin_first_zero=False,  # se True e phase=="after" e num_meetings==2 => (0, x)
                          debug=False):
    """
    Deduplica i path scegliendo una sola tupla per ciascuna somma.
    - Se cut_values è lista di tuple (già path precomposti) della giusta lunghezza, li accetta così come sono.
    - Altrimenti genera product(cut_values, repeat=num_meetings) e deduplica per somma.
    Opzioni:
      - phase="after" & pin_first_zero=True & num_meetings==2  -> forza pattern (0, x)
      - switch_bps: per somme <= soglia (in bps) usa right-bias (ultima tupla).
    """
    if not cut_values:
        return []

    # Se l'utente ha già costruito i path (lista di tuple), accettali e deduplica preservando ordine
    if isinstance(cut_values[0], (tuple, list)):
        ready = [tuple(p) for p in cut_values if len(p) == num_meetings]
        return list(dict.fromkeys(ready))

    # Caso speciale: voglio (0, x) quando ho 2 meeting "in-exp"
    if phase == "after" and pin_first_zero and num_meetings == 2:
        return [(0.0, v) for v in sorted(cut_values)]

    scale = 100 if units == "percent" else 10000
    vals = sorted(cut_values)

    buckets = defaultdict(list)
    for path in itertools.product(vals, repeat=num_meetings):
        key_bps = int(round(sum(path) * scale))  # chiave intera in bps per stabilità numerica
        buckets[key_bps].append(path)

    out = []
    for key_bps, paths in buckets.items():
        if phase == "after" and switch_bps is not None and num_meetings == 2 and key_bps <= switch_bps:
            # right-biased: preferisci l'ultima combinazione (spinge il taglio sul meeting più tardo)
            out.append(paths[-1])
        else:
            # default: left-biased (prima combinazione), identico al tuo comportamento originale
            out.append(paths[0])

    if debug:
        print(f"[unique_paths_by_sum] buckets={len(buckets)}  example={out[:5]} ...")
    return out

###################################################################################################
def generate_fv_grid(
    eff_dt_list, 
    meetings_before_fut_life, 
    meetings_inside_fut_life,
    compound_start, 
    compound_end, 
    fut_start_dt,
    OIS_df, 
    TARGET2_HOLIDAYS, 
    cut_values_to_meeting, 
    cut_values_after_fut_start, 
    landing_zone_func, 
    basis, 
    n_jobs = -1
):
    #Cartesian prod for rate paths
    #rates_path_to_meeting = list(itertools.product(cut_values_to_meeting, repeat = len(meetings_before_fut_life)))
    #rates_path_after_fut_start = list(itertools.product(cut_values_after_fut_start, repeat = len(meetings_inside_fut_life)))
    
    rates_path_to_meeting = unique_paths_by_sum(cut_values_to_meeting, len(meetings_before_fut_life))
    rates_path_after_fut_start = unique_paths_by_sum(cut_values_after_fut_start, len(meetings_inside_fut_life))
    
    
    #Worker function to process each row:
    def compute_row(path_after):
        row = []
        for path_to in rates_path_to_meeting:
            pair = list(path_to) + list(path_after)
            tmp = landing_zone_func(
                TARGET2_HOLIDAYS,
                compound_start, 
                compound_end, 
                OIS_df, 
                eff_dt_list,
                pair, 
                fut_start_dt
            )
            row.append(tmp.iloc[0].values[0])
        return row
    
    #Parallel execution
    data_matrix = Parallel(n_jobs = n_jobs)(
        delayed(compute_row)(path_after) for path_after in rates_path_after_fut_start
    )
    
    #Create df
    df_final = pd.DataFrame(
        data_matrix,
        index = [sum(path) for path in rates_path_after_fut_start],
        columns = [sum(path) for path in rates_path_to_meeting]
    )
    
    df_final = df_final + basis
    df_final = 100 - df_final
    df_final.columns = [round(float(col), 2) for col in df_final.columns]
    df_final.index = [round(float(col), 2) for col in df_final.index]
    
    df_final = df_final.loc[~df_final.index.duplicated(), ~df_final.columns.duplicated()]
    
    return df_final


######## Wrapper function that compute landing zone per contract adjust directly for number of meetings inside fut.life#####

def contract_pricer_fv(contract_code, df_euribors, index_name, col_name,
                  unit=100, min_bp=-25, max_bp=25,
                  min_bp_meeting_to_isolate=-25, max_bp_meeting_to_isolate=25):

    #### General inputs:
    DIV = unit
    MIN_BPS, MAX_BPS = min_bp, max_bp
    MIN_BPS_ISOLATED_MEETING, MAX_BPS_ISOLATED_MEETING = min_bp_meeting_to_isolate, max_bp_meeting_to_isolate

    STEP_CUMUL = 5    ### Meeting before step in bp
    STEP_ISOLATED = 1 ### Meeting to isolate step in bp

    ## Pulling Contract specs:
    fut_start_dt, fut_end_dt, eff_dt_list, before, inside, basis = pull_inputs(df_euribors, contract_code)
    eff_dt_list  = sorted(pd.to_datetime(eff_dt_list))
    fut_start_dt = pd.Timestamp(fut_start_dt)
    fut_end_dt   = pd.Timestamp(fut_end_dt)
    compound_start = eff_dt_list[0]
    compound_end   = pd.Timestamp(fut_end_dt)

    print("to_exp (<= start):", [d.strftime("%Y-%m-%d") for d in before])
    print("in_exp (>= start):", [d.strftime("%Y-%m-%d") for d in inside])
    print("effective_meetings_list ->", [d.strftime("%Y-%m-%d") for d in eff_dt_list])

    if len(inside) > 1:
        print('More than one meeting after future expiry')

        ### Rates paths creation:
        vals_to = [x / DIV for x in range(MIN_BPS, MAX_BPS + 1, STEP_CUMUL)]
        vals_first_meeting_after_fut_start = [x / DIV for x in range(MIN_BPS, MAX_BPS + 1, STEP_CUMUL)]
        vals_isolated_meeting = [x / DIV for x in range(MIN_BPS_ISOLATED_MEETING, MAX_BPS_ISOLATED_MEETING + 1, STEP_ISOLATED)]

        to_paths_inside = unique_paths_by_sum(vals_to, num_meetings=len(before))

        # ================== Initialize grid  ==================
        idx_levels_bps = sorted({
            int(round((sum(t) + m) * DIV))
            for t in to_paths_inside
            for m in vals_first_meeting_after_fut_start
        })
        cols_bps = [int(round(a * DIV)) for a in vals_isolated_meeting]
        grid_fv = pd.DataFrame(index=idx_levels_bps, columns=cols_bps, dtype=float)

        # ================== Compute FV grid ==================
        for to_tuple in to_paths_inside:
            s_to = sum(to_tuple)
            for m in vals_first_meeting_after_fut_start:
                after_paths = [(m, a) for a in vals_isolated_meeting]

                grid_vec = generate_fv_grid(
                    eff_dt_list,
                    before,
                    inside,
                    compound_start,
                    compound_end,
                    fut_start_dt,
                    ESTR_DF,
                    TARGET2_HOLIDAYS,
                    [to_tuple],
                    after_paths,
                    landing_zone_func=euribor_landing_zone,
                    basis=basis,
                    n_jobs=-1
                )

                # Extract prices
                count = min(grid_vec.shape[0], len(after_paths))
                prices = grid_vec.iloc[:count, 0].astype(float).tolist()

                key_bps = int(round((s_to + m) * DIV))
                for j in range(count):
                    a_bps = cols_bps[j]
                    grid_fv.at[key_bps, a_bps] = prices[j]

        grid_fv.index.name = index_name
        grid_fv.columns.name = col_name
        grid_fv = grid_fv.sort_index()

    else:
        print("One meeting after future expiry")

        # --- bp vals ---
        vals_to = [x / DIV for x in range(MIN_BPS, MAX_BPS + 1, STEP_CUMUL)]
        vals_isolated_meeting = [x / DIV for x in range(MIN_BPS_ISOLATED_MEETING,
                                                        MAX_BPS_ISOLATED_MEETING + 1, STEP_ISOLATED)]

        to_paths_inside = unique_paths_by_sum(vals_to, num_meetings=len(before))

        # bp vals
        rows_bps = [int(round(a * DIV)) for a in vals_isolated_meeting]        
        cols_bps = [int(round(sum(t) * DIV)) for t in to_paths_inside]       

         # ================== Initialize grid  ==================
        grid_fv = pd.DataFrame(index=rows_bps, columns=cols_bps, dtype=float)

        after_paths = [(a,) for a in vals_isolated_meeting]

        for col_idx, to_tuple in enumerate(to_paths_inside):
            grid_vec = generate_fv_grid(
                eff_dt_list,
                before,
                inside,
                compound_start,
                compound_end,
                fut_start_dt,
                ESTR_DF,
                TARGET2_HOLIDAYS,
                [to_tuple],
                after_paths,
                landing_zone_func=euribor_landing_zone,
                basis=basis,
                n_jobs=-1
            )

            count = min(grid_vec.shape[0], len(after_paths))
            prices = grid_vec.iloc[:count, 0].astype(float).tolist()
            for i in range(count):
                grid_fv.iat[i, col_idx] = prices[i]
                
        grid_fv = grid_fv.T
        grid_fv.index.name = index_name
        grid_fv.columns.name = col_name
        grid_fv = grid_fv.sort_index()

    return grid_fv

################ VISUALS ###############################
def filter_landing_zone(df, row_min , row_max, 
                        col_min, col_max):
    
    row_mask = (df.index.values >= row_min) & (df.index.values <= row_max)
    col_mask = (df.columns.values >= col_min) & (df.columns.values <= col_max)
    return df.loc[row_mask, col_mask].copy()


###### Heatmap #########################
def plot_landing_zone_heatmap(
    df,
    title="Landing Zone Heatmap",
    x_title="X (bps)",
    y_title="Y (bps)",
    colorscale="RdYlGn",
    decimals=3,
    text_size=11,
    height=900,
    reverse_y=True,
    show_all_ticks=True,   # <-- TOGGLE
    tick_step=5,           # se show_all_ticks=False, mostra 1 tick ogni tick_step bp
):
    x = df.columns.values
    y = df.index.values

    fig = go.Figure(data=go.Heatmap(
        z=df.values,
        x=x,
        y=y,
        colorscale=colorscale,
        text=np.round(df.values, decimals),
        texttemplate="%{text}",
        textfont=dict(size=text_size),
        hovertemplate=(
            f"{x_title.replace(' (bps)','')}: %{{x}} bp<br>"
            f"{y_title.replace(' (bps)','')}: %{{y}} bp<br>"
            f"Value: %{{z:.{decimals}f}}<extra></extra>"
        ),
        colorbar=dict(title="Value")
    ))

    # --- ticks ---
    if show_all_ticks:
        x_tickvals = x
        y_tickvals = y
    else:
        # tiene solo i tick multipli di tick_step (funziona bene se x/y sono interi bp)
        x_tickvals = [v for v in x if (float(v) % tick_step) == 0]
        y_tickvals = [v for v in y if (float(v) % tick_step) == 0]

    fig.update_layout(
        title=title,
        xaxis=dict(
            title=x_title,
            tickmode="array",
            tickvals=x_tickvals,
            ticktext=[str(v) for v in x_tickvals],
        ),
        yaxis=dict(
            title=y_title,
            tickmode="array",
            tickvals=y_tickvals,
            ticktext=[str(v) for v in y_tickvals],
            autorange="reversed" if reverse_y else True
        ),
        height=height
    )

    return fig 


# NOTEBOOK STARTS HERE:

## 1.) Collecting ECB's Meetings,  holiday schedule for used for compounding and effective dates

### ECB Meetings Calendars - (MANUAL INPUT):


In [ ]:
### ECB Meetings for 2025 and 2026- Manual input:

ECB_calendar = [
    
    dt.datetime(2025, 1, 30), dt.datetime(2025, 3, 6), dt.datetime(2025, 4, 17), dt.datetime(2025, 6, 5), 
    dt.datetime(2025, 7, 24), dt.datetime(2025, 9, 11), dt.datetime(2025, 10, 30), dt.datetime(2025, 12, 18), 
    
    dt.datetime(2026, 2, 5), dt.datetime(2026, 3, 19), dt.datetime(2026,4, 30), 
    dt.datetime(2026, 6, 11), dt.datetime(2026, 7, 23), dt.datetime(2026, 9, 10), 
    dt.datetime(2026, 10, 29), dt.datetime(2026, 12, 17)
    
]

### Target 2 Holidays - (MANUAL INPUT):

In [ ]:
### Source: https://www.ecb.europa.eu/ecb/contacts/working-hours/html/index.en.html

TARGET2_HOLIDAYS = [
    
    '2025-01-01', '2025-04-18', '2025-04-21', '2025-05-01', '2025-05-09', '2025-05-29', '2025-06-09','2025-06-19',
    '2025-10-03', '2025-11-01', '2025-12-24', '2025-12-25', '2025-12-26', '2025-12-31', 
    
    '2026-01-01', '2026-04-03', '2026-04-06', '2026-05-01', '2026-05-09', '2026-05-06', '2026-05-17','2026-05-27',
    '2026-10-03', '2026-11-01', '2026-12-24', '2026-12-25', '2026-12-26', '2026-12-31',
    
    '2027-01-01', '2027-03-26', '2027-03-29', '2027-05-01', '2027-05-09', '2027-05-06', '2027-05-17','2027-05-27',
    '2027-10-03', '2027-11-01', '2027-12-24', '2027-12-25', '2027-12-26', '2027-12-31'
]


### For adjustment in dataframe creation
target_2_holidays_for_check = [pd.Timestamp(d) for d in TARGET2_HOLIDAYS]

In [ ]:
target2_bday = CustomBusinessDay(holidays = TARGET2_HOLIDAYS)

### ECB Meetings and Effective dates:

In [ ]:
prefix = 'EZ0BFR '
suffix = ' Index'

## Next ECB Meetings:
next_ECB_meetings = [meeting for meeting in ECB_calendar if meeting >= dt.datetime.today()]
next_ECB_meetings_dic = {prefix+ calendar.month_abbr[i.month].upper()+ str(i.year)+ suffix: i for i in next_ECB_meetings}

## Previous ECB Meetings:

prev_ECB_meetings= [meeting for meeting in ECB_calendar if meeting <= dt.datetime.today()]
prev_ECB_meetings_dic = {prefix+ calendar.month_abbr[i.month].upper()+ str(i.year)+ suffix: i for i in prev_ECB_meetings}

### Adding Effective dates:

ecb_meetings = pd.DataFrame([next_ECB_meetings_dic], index = ['Meeting_dates']).T
ecb_meetings['Effective_date'] = ecb_meetings['Meeting_dates'] + pd.Timedelta(days = 6)
ecb_meetings['Days_diff_between_Eff_dt'] = ecb_meetings['Effective_date'].diff().dt.days

ecb_meetings = adjust_effective_Date(ecb_meetings, target_2_holidays_for_check)


prev_ecb_meetings = pd.DataFrame([prev_ECB_meetings_dic], index = ['Meeting_dates']).T
prev_ecb_meetings['Effective_date'] = prev_ecb_meetings['Meeting_dates'] + pd.Timedelta(days = 6)
prev_ecb_meetings['Days_diff_between_Eff_dt'] = prev_ecb_meetings['Effective_date'].diff().dt.days

prev_ecb_meetings = adjust_effective_Date(ecb_meetings, target_2_holidays_for_check)


all_ECB_meetings = pd.concat([prev_ecb_meetings, ecb_meetings], axis = 0)
all_ECB_meetings = all_ECB_meetings.drop(columns= ['Days_diff_between_Eff_dt'])

# 2.) Euribors Futures:

In [ ]:
date = dt.datetime.today()
fld = "PX_LAST"

### Data download:

In [ ]:
## Create tickers:
euribors = er_contracts_generation(date)

### Fields:
flds = ['LAST_TRADEABLE_DT', 'INT_RATE_FUT_START_DT', 'INT_RATE_FUT_END_DT']
cols_euribors = ['last_trading_dd', 'future_start_dt', 'future_end_dt']

## Data retrival:
df_euribors = pd.DataFrame()

for f in flds:
    tmp = bdp(euribors, f)
    df_euribors = pd.concat([df_euribors, tmp], axis = 1)

df_euribors.columns = cols_euribors

### Adding dd lenght to overall contract lenght once future is expired:
df_euribors['future_period_dd'] = (df_euribors['future_end_dt'] - df_euribors['future_start_dt']).dt.days


df_euribors

### Adding Effective dates and weight of the meetings falling inside the ER futures life:

In [ ]:
bef_exp, aft_exp, contr_weight = meetings_ranges(ecb_meetings, df_euribors, verbose = True)

In [ ]:
df_euribors['Eff_date_to_exp'], df_euribors['Eff_date_after_exp'],  df_euribors['ECB_after_expiry_weight_%'] = bef_exp, aft_exp, contr_weight

df_euribors = df_euribors.T 
df_euribors

## 2.b) Euribor-ESTR Basis:

#### Downloading Euribor-ESTR bais and add the value of the basis to the corresponding future contract:

In [ ]:
basis_tickers = ['TKY' + c for c in list(df_euribors.columns)]

df_basis = pd.DataFrame()
for i in basis_tickers:
    tmp = bdp(i, 'PX_LAST')
    df_basis = pd.concat([df_basis, tmp], axis = 0)

df_basis.index.rename('Euribor-ESTR basis', inplace = True)

## Append Euribor-Estr to the euribors dataframe:
basis_series = df_basis.loc[:, 'PX_LAST']
basis_series.index = [ticker.replace('TKY',  '') for ticker in basis_series.index]

basis_series = basis_series.reindex(list(df_euribors.columns))

df_euribors.loc['Euribor-ESTR basis'] = basis_series.values 

df_euribors

# 3. Downloading ESTR Overnight:

#### Data download Inputs:

In [ ]:
from_ = dt.datetime(2025, 1,1)
to_ = dt.datetime.today()

fld = ['PX_LAST']

#### Data download:

In [ ]:
col = ['ESTR_ON_Rate']

ESTR_DF = bbg_download_clean_df(['ESTRON Index'], fld, from_, to_)
ESTR_DF.columns = col

ESTR_DF.head(10)

# 4. Fair Value Surface Grid Generation:

## FIRST CONTRACT:

#### Inputs:

In [ ]:
first_contract = 'ERH6 Comdty' ## will need to be changed once contract expires

## Defining columns names for the grid:
index_name   = " To March (bps)"
cols_name  = " April pricing (bps)"

bp_min, bp_max = -25, +25
bp_min_isolated_meeting, bp_max_isolated_meeting = -25, +25

### Landing Zones grid:

In [ ]:
ER1 = contract_pricer_fv(
    first_contract, 
    df_euribors, 
    index_name, 
    cols_name, 
    unit = 100,
    min_bp = bp_min,
    max_bp = bp_max, 
    min_bp_meeting_to_isolate= bp_min_isolated_meeting,
    max_bp_meeting_to_isolate= bp_max_isolated_meeting
)

ER1 = ER1.round(3)

In [ ]:
## Full Table:
ER1.head(10)

#### Visuals:

In [ ]:
###### DISPLAY FINAL TABLE  #####:
row_min_bp_range , row_max_bp_range = -25, 25 
col_min_bp_range , col_max_bp_range = -25, 0

title = "ERH6 landing grid"
x_title = 'To March'
y_title = "To April"

ER1_filtered = filter_landing_zone(ER1, row_min_bp_range, row_max_bp_range,
                                   col_min_bp_range, col_max_bp_range).T

plot_landing_zone_heatmap(ER1_filtered, title = title, x_title = x_title, y_title = y_title)



## SECOND CONTRACT:

In [ ]:
second_contract = 'ERM6 Comdty' ## will need to be changed once contract expires

## Defining columns names for the grid:
index_name   = "To June"
cols_name  = " To July"

bp_min, bp_max = -25, +25
bp_min_isolated_meeting, bp_max_isolated_meeting = -25, +25


In [ ]:
ER2 = contract_pricer_fv(
    second_contract, 
    df_euribors, 
    index_name, 
    cols_name, 
    unit = 100,
    min_bp = bp_min,
    max_bp = bp_max, 
    min_bp_meeting_to_isolate= bp_min_isolated_meeting,
    max_bp_meeting_to_isolate= bp_max_isolated_meeting
)

ER2 = ER2.round(3)

In [ ]:
ER2.head(10)

#### Visuals:

## THIRD CONTRACT:

In [ ]:
third_contract = 'ERU6 Comdty' ## will need to be changed once contract expires

## Defining columns names for the grid:
index_name   = "To Sep"
cols_name  = " To Oct"

bp_min, bp_max = -25, +25
bp_min_isolated_meeting, bp_max_isolated_meeting = -25, +25


FV Grid:

In [ ]:
ER3 = contract_pricer_fv(
    third_contract, 
    df_euribors, 
    index_name, 
    cols_name, 
    unit = 100,
    min_bp = bp_min,
    max_bp = bp_max, 
    min_bp_meeting_to_isolate= bp_min_isolated_meeting,
    max_bp_meeting_to_isolate= bp_max_isolated_meeting
)

ER3 = ER3.round(3)

In [ ]:
ER3.head(10)

#### Visuals:

In [ ]:
###### DISPLAY FINAL TABLE  #####:
row_min_bp_range , row_max_bp_range = -25, 25 
col_min_bp_range , col_max_bp_range = -25, 25

title = "ERU6 landing grid"
x_title = 'To June'
y_title = "To July"

ER3_filtered = filter_landing_zone(ER3, -25, 25, 0, 25).T

plot_landing_zone_heatmap(ER3_filtered, title = title, x_title = x_title, y_title = y_title)

# FOURTH CONTRACT:

In [ ]:
fourth_contract = 'ERZ6 Comdty' ## will need to be changed once contract expires

## Defining columns names for the grid:
index_name   = "Cumulative to Dec (bps)"
cols_name  = " Mar pricing (bps)"

bp_min, bp_max = -25, +25
bp_min_isolated_meeting, bp_max_isolated_meeting = -25, +25

3. FV GRID

In [ ]:
ER4 = contract_pricer_fv(
    fourth_contract, 
    df_euribors, 
    index_name, 
    cols_name, 
    unit = 100,
    min_bp = bp_min,
    max_bp = bp_max, 
    min_bp_meeting_to_isolate= bp_min_isolated_meeting,
    max_bp_meeting_to_isolate= bp_max_isolated_meeting
)

ER4 = ER4.round(3)

In [ ]:
###### DISPLAY FINAL TABLE  #####:
row_min_bp_range , row_max_bp_range = -25, 25 
col_min_bp_range , col_max_bp_range = -25, 25

title = "ERZ6 landing grid"
x_title = 'To Dec'
y_title = "To March"

ER4_filtered = filter_landing_zone(ER4, -25, 25, 0, 25).T

plot_landing_zone_heatmap(ER4_filtered, title = title, x_title = x_title, y_title = y_title)

# 5.) DATA EXPORT TO EXCEL:

In [ ]:
df_euribors_formatted = df_euribors.copy()
df_euribors_formatted = df_euribors_formatted.T


date_cols = ['last_trading_dd','future_start_dt', 'future_end_dt', 'Eff_date_to_exp', 'Eff_date_after_exp']

for col in date_cols:
    df_euribors_formatted[col] = df_euribors_formatted[col].apply(
    lambda x: ', '.join(pd.to_datetime(x).strftime('%#d-%b-%y') if isinstance(x, pd.Timestamp)
                       else [pd.to_datetime(i).strftime('%#d-%b-%y') for i in x]) 
        if isinstance(x, list) else pd.to_datetime(x).strftime('%#d-%b-%y'))
    
    
df_euribors_formatted['ECB_after_expiry_weight_%'] =  df_euribors_formatted['ECB_after_expiry_weight_%'].apply(
    lambda x: ', '.join([f"{round(float(i)*100, 1)}%" for i in x]) if isinstance(x, list)
     else f"{round(float(x)*100, 1)}%"
)

df_euribors_formatted['Euribor-ESTR basis'] = [
    (i*100) for i in df_euribors_formatted['Euribor-ESTR basis']
]


df_euribors_formatted = df_euribors_formatted.rename(columns = {
    'last_trading_dd': 'last trading day',
    'future_start_dt': 'underlying start',
    'future_end_dt': 'underlying ends',
    'future_period_dd': 'lenght (dd)',
    'Eff_date_to_exp': 'Eff.dates before fut. expiry',
    'Eff_date_after_exp': 'Eff.dates inside fut. life',
    'ECB_after_expiry_weight_%': 'Weights of meetings inside fut.life',
    'Euribor-ESTR basis': 'Euribor-ESTR basis (bp)' 
})

                                         
df_euribors_formatted = df_euribors_formatted.T

## Final Formatted Contract specs table:
df_euribors_formatted

### Export Everything:

In [ ]:
path_export = "L\Front_End\\Euribor Fair Value Grids\\Export From pyhton\Euribors.xlsx"

In [ ]:
with pd.ExcelWriter(path = path_export) as writer:
    df_euribors_formatted.to_excel(writer, sheet_name  = 'Contract_specs')
    ER1.to_excel(writer, sheet_name= 'ER1')
    ER2.to_excel(writer, sheet_name= 'ER2')
    ER3.to_excel(writer, sheet_name= 'ER3')
    ER4.to_excel(writer, sheet_name= 'ER4')
    #fifth_contract_contract_grid.to_excel(writer, sheet_name= 'ER5')
    